In [0]:
%run ../99_utils/utils_config

In [0]:
%run ../99_utils/utils_logger

PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: c1a8a41c-306d-41c7-9851-786d60a7e41b


In [0]:
%run ../99_utils/utils_table_logger

utils_config loaded successfully.


utils_logger loaded successfully.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 Job — Run Full Pipeline
# MAGIC
# MAGIC Orchestrates the complete execution flow of the Brazil Legislative Analytics Medallion project.
# MAGIC
# MAGIC ## Purpose
# MAGIC This notebook executes the project pipeline in the expected order, from setup validation to quality checks.
# MAGIC
# MAGIC ## Execution Flow
# MAGIC - Validate project setup
# MAGIC - Optionally validate API connection
# MAGIC - Execute Bronze ingestion notebooks
# MAGIC - Execute Silver transformation notebooks
# MAGIC - Execute Gold dimensional modeling notebooks
# MAGIC - Execute Marts notebooks
# MAGIC - Execute Quality notebooks
# MAGIC
# MAGIC ## Current Development Note
# MAGIC Some Bronze, Silver, Gold and Marts notebooks may still be under development.
# MAGIC In this case, the corresponding execution blocks can remain disabled until the notebooks are available.
# MAGIC
# MAGIC ## API Validation Note
# MAGIC The API validation notebook is intentionally disabled by default in the full pipeline because external API endpoints may be unstable or slow in Databricks Free Edition.
# MAGIC It should be executed manually or as a separated job when deep API validation is required.
# MAGIC
# MAGIC ## Documentation Standard
# MAGIC - Python functions and variables are written in English.
# MAGIC - Table and field names follow Portuguese mnemonic standards.
# MAGIC - Comments and documentation are written in English.

# COMMAND ----------

# MAGIC %run ../99_utils/utils_config

# COMMAND ----------

# MAGIC %run ../99_utils/utils_logger

# COMMAND ----------

# MAGIC %run ../99_utils/utils_table_logger

# COMMAND ----------

from datetime import datetime
import time
import uuid
import traceback

# COMMAND ----------

print("=" * 90)
print("BRAZIL LEGISLATIVE ANALYTICS MEDALLION")
print("01 - RUN FULL PIPELINE")
print("=" * 90)
print(f"Execution Timestamp: {datetime.now()}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Environment: {PROJECT_ENVIRONMENT}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# JOB CONFIGURATION
# ============================================================

NOTEBOOK_NAME = "01_run_full_pipeline"
LAYER_NAME = "jobs"
ENTITY_NAME = "full_pipeline"

JOB_RUN_ID = str(uuid.uuid4())

# During development, keep this as False to avoid stopping
# the full job when optional or not-yet-built notebooks are missing.
FAIL_ON_STEP_ERROR = False

PIPELINE_LOG_TABLE = (
    f"{CATALOG_NAME}."
    f"{SCHEMA_AUDIT}."
    f"{AUD_TB_LOG_EXECUCAO_PIPELINE}"
)

PIPELINE_ERROR_TABLE = (
    f"{CATALOG_NAME}."
    f"{SCHEMA_AUDIT}."
    f"{AUD_TB_LOG_ERROS_PIPELINE}"
)

logger = get_logger(
    logger_name=NOTEBOOK_NAME,
    layer_name=LAYER_NAME,
)

# COMMAND ----------

# ============================================================
# PIPELINE STEP CONFIGURATION
# ============================================================
#
# Set enabled=True only when the referenced notebook exists and
# is ready to execute.
#
# API validation is disabled by default in the full pipeline
# because the external API may be slow or unstable.
#
# ============================================================

PIPELINE_STEPS = [
    {
        "step_name": "validate_project_setup",
        "notebook_path": "../00_setup/90_validate_project_setup",
        "layer_name": "setup",
        "entity_name": "project_setup",
        "enabled": True,
        "critical": True,
    },
    {
        "step_name": "validate_api_connection",
        "notebook_path": "../00_setup/92_validate_api_connection",
        "layer_name": "setup",
        "entity_name": "api_connection",
        "enabled": True,
        "critical": True,
    },

    # Bronze ingestion notebooks.
    {
        "step_name": "bronze_deputados",
        "notebook_path": "../01_bronze/01_bronze_deputados",
        "layer_name": "bronze",
        "entity_name": "deputados",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "bronze_frentes",
        "notebook_path": "../01_bronze/02_bronze_frentes",
        "layer_name": "bronze",
        "entity_name": "frentes",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "bronze_frentes_membros",
        "notebook_path": "../01_bronze/03_bronze_frentes_membros",
        "layer_name": "bronze",
        "entity_name": "frentes_membros",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "bronze_frentes_detalhes",
        "notebook_path": "../01_bronze/03a_bronze_frentes_detalhes",
        "layer_name": "bronze",
        "entity_name": "frentes_detalhes",
        "enabled": False,
        "critical": False,
    },
    {
        "step_name": "bronze_eventos",
        "notebook_path": "../01_bronze/04_bronze_eventos",
        "layer_name": "bronze",
        "entity_name": "eventos",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "bronze_votacoes",
        "notebook_path": "../01_bronze/05_bronze_votacoes",
        "layer_name": "bronze",
        "entity_name": "votacoes",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "bronze_votos",
        "notebook_path": "../01_bronze/06_bronze_votos",
        "layer_name": "bronze",
        "entity_name": "votos",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "bronze_despesas_ceap",
        "notebook_path": "../01_bronze/07_bronze_despesas_ceap",
        "layer_name": "bronze",
        "entity_name": "despesas_ceap",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "bronze_cpis",
        "notebook_path": "../01_bronze/08_bronze_cpis",
        "layer_name": "bronze",
        "entity_name": "cpis",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "bronze_cpi_eventos",
        "notebook_path": "../01_bronze/09_bronze_cpi_eventos",
        "layer_name": "bronze",
        "entity_name": "cpi_eventos",
        "enabled": False,
        "critical": False,
    },
    {
        "step_name": "bronze_proposicoes",
        "notebook_path": "../01_bronze/10_bronze_proposicoes",
        "layer_name": "bronze",
        "entity_name": "proposicoes",
        "enabled": False,
        "critical": True,
    },

    # Silver transformation notebooks.
    {
        "step_name": "silver_deputados",
        "notebook_path": "../02_silver/01_silver_deputados",
        "layer_name": "silver",
        "entity_name": "deputados",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_partidos",
        "notebook_path": "../02_silver/02_silver_partidos",
        "layer_name": "silver",
        "entity_name": "partidos",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_estados",
        "notebook_path": "../02_silver/03_silver_estados",
        "layer_name": "silver",
        "entity_name": "estados",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_frentes",
        "notebook_path": "../02_silver/04_silver_frentes",
        "layer_name": "silver",
        "entity_name": "frentes",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_frentes_membros",
        "notebook_path": "../02_silver/05_silver_frentes_membros",
        "layer_name": "silver",
        "entity_name": "frentes_membros",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_eventos",
        "notebook_path": "../02_silver/06_silver_eventos",
        "layer_name": "silver",
        "entity_name": "eventos",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_votacoes",
        "notebook_path": "../02_silver/07_silver_votacoes",
        "layer_name": "silver",
        "entity_name": "votacoes",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_votos",
        "notebook_path": "../02_silver/08_silver_votos",
        "layer_name": "silver",
        "entity_name": "votos",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_despesas_ceap",
        "notebook_path": "../02_silver/09_silver_despesas_ceap",
        "layer_name": "silver",
        "entity_name": "despesas_ceap",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_fornecedores",
        "notebook_path": "../02_silver/10_silver_fornecedores",
        "layer_name": "silver",
        "entity_name": "fornecedores",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_cpis",
        "notebook_path": "../02_silver/11_silver_cpis",
        "layer_name": "silver",
        "entity_name": "cpis",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_cpi_eventos",
        "notebook_path": "../02_silver/12_silver_cpi_eventos",
        "layer_name": "silver",
        "entity_name": "cpi_eventos",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "silver_proposicoes",
        "notebook_path": "../02_silver/13_silver_proposicoes",
        "layer_name": "silver",
        "entity_name": "proposicoes",
        "enabled": False,
        "critical": True,
    },

    # Gold dimensional modeling notebooks.
    {
        "step_name": "gold_dm_deputado",
        "notebook_path": "../03_gold/01_dm_deputado",
        "layer_name": "gold",
        "entity_name": "dm_deputado",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_partido",
        "notebook_path": "../03_gold/02_dm_partido",
        "layer_name": "gold",
        "entity_name": "dm_partido",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_estado",
        "notebook_path": "../03_gold/03_dm_estado",
        "layer_name": "gold",
        "entity_name": "dm_estado",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_data",
        "notebook_path": "../03_gold/04_dm_data",
        "layer_name": "gold",
        "entity_name": "dm_data",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_orgao",
        "notebook_path": "../03_gold/05_dm_orgao",
        "layer_name": "gold",
        "entity_name": "dm_orgao",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_tipo_evento",
        "notebook_path": "../03_gold/06_dm_tipo_evento",
        "layer_name": "gold",
        "entity_name": "dm_tipo_evento",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_evento",
        "notebook_path": "../03_gold/07_dm_evento",
        "layer_name": "gold",
        "entity_name": "dm_evento",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_votacao",
        "notebook_path": "../03_gold/08_dm_votacao",
        "layer_name": "gold",
        "entity_name": "dm_votacao",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_tipo_votacao",
        "notebook_path": "../03_gold/09_dm_tipo_votacao",
        "layer_name": "gold",
        "entity_name": "dm_tipo_votacao",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_frente",
        "notebook_path": "../03_gold/10_dm_frente",
        "layer_name": "gold",
        "entity_name": "dm_frente",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_fornecedor",
        "notebook_path": "../03_gold/11_dm_fornecedor",
        "layer_name": "gold",
        "entity_name": "dm_fornecedor",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_dm_cpi",
        "notebook_path": "../03_gold/12_dm_cpi",
        "layer_name": "gold",
        "entity_name": "dm_cpi",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_ft_frentes_membros",
        "notebook_path": "../03_gold/13_ft_frentes_membros",
        "layer_name": "gold",
        "entity_name": "ft_frentes_membros",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_ft_eventos_presencas",
        "notebook_path": "../03_gold/14_ft_eventos_presencas",
        "layer_name": "gold",
        "entity_name": "ft_eventos_presencas",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_ft_resultados_votacoes",
        "notebook_path": "../03_gold/15_ft_resultados_votacoes",
        "layer_name": "gold",
        "entity_name": "ft_resultados_votacoes",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_ft_despesas_ceap",
        "notebook_path": "../03_gold/16_ft_despesas_ceap",
        "layer_name": "gold",
        "entity_name": "ft_despesas_ceap",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "gold_ft_cpi_eventos",
        "notebook_path": "../03_gold/17_ft_cpi_eventos",
        "layer_name": "gold",
        "entity_name": "ft_cpi_eventos",
        "enabled": False,
        "critical": True,
    },

    # Marts.
    {
        "step_name": "mart_atlas_frentes",
        "notebook_path": "../04_marts/01_am_atlas_frentes",
        "layer_name": "marts",
        "entity_name": "atlas_frentes",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "mart_calendario_eventos",
        "notebook_path": "../04_marts/02_am_calendario_eventos",
        "layer_name": "marts",
        "entity_name": "calendario_eventos",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "mart_correlacao_frentes_votacoes",
        "notebook_path": "../04_marts/03_am_correlacao_frentes_votacoes",
        "layer_name": "marts",
        "entity_name": "correlacao_frentes_votacoes",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "mart_panorama_despesas_ceap",
        "notebook_path": "../04_marts/04_am_panorama_despesas_ceap",
        "layer_name": "marts",
        "entity_name": "panorama_despesas_ceap",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "mart_auditoria_cpis",
        "notebook_path": "../04_marts/05_am_auditoria_cpis",
        "layer_name": "marts",
        "entity_name": "auditoria_cpis",
        "enabled": False,
        "critical": True,
    },
    {
        "step_name": "mart_monitor_presenca_absenteismo",
        "notebook_path": "../04_marts/06_am_monitor_presenca_absenteismo",
        "layer_name": "marts",
        "entity_name": "monitor_presenca_absenteismo",
        "enabled": False,
        "critical": True,
    },

    # Quality notebooks.
    {
        "step_name": "quality_bronze_checks",
        "notebook_path": "../06_quality/01_quality_bronze_checks",
        "layer_name": "quality",
        "entity_name": "bronze_checks",
        "enabled": True,
        "critical": False,
    },
    {
        "step_name": "quality_silver_checks",
        "notebook_path": "../06_quality/02_quality_silver_checks",
        "layer_name": "quality",
        "entity_name": "silver_checks",
        "enabled": True,
        "critical": False,
    },
    {
        "step_name": "quality_gold_checks",
        "notebook_path": "../06_quality/03_quality_gold_checks",
        "layer_name": "quality",
        "entity_name": "gold_checks",
        "enabled": True,
        "critical": False,
    },
    {
        "step_name": "quality_traceability_checks",
        "notebook_path": "../06_quality/04_traceability_checks",
        "layer_name": "quality",
        "entity_name": "traceability_checks",
        "enabled": True,
        "critical": False,
    },
]

# COMMAND ----------

# ============================================================
# JOB HELPERS
# ============================================================

def run_notebook_step(
    step_config: dict,
) -> dict:
    """
    Executes a configured notebook step and returns execution metadata.
    """

    step_name = step_config["step_name"]
    notebook_path = step_config["notebook_path"]
    step_layer = step_config["layer_name"]
    step_entity = step_config["entity_name"]
    is_enabled = step_config["enabled"]
    is_critical = step_config["critical"]

    step_log_id = str(uuid.uuid4())
    started_at = datetime.now()
    status = EXECUTION_STATUS_STARTED
    message = "Step execution started."

    if not is_enabled:

        finished_at = datetime.now()
        duration_seconds = (
            finished_at - started_at
        ).total_seconds()

        write_pipeline_log(
            log_id=step_log_id,
            execution_id=JOB_RUN_ID,
            notebook_name=NOTEBOOK_NAME,
            layer_name=step_layer,
            entity_name=step_entity,
            target_table="not_applicable",
            status=EXECUTION_STATUS_WARNING,
            message=f"Step skipped because it is disabled: {step_name}",
            started_at=started_at,
            finished_at=finished_at,
            duration_seconds=duration_seconds,
            records_read=None,
            records_written=None,
        )

        log_warning(
            pipeline_logger=logger,
            message=f"Step skipped: {step_name}",
        )

        return {
            "step_name": step_name,
            "status": EXECUTION_STATUS_WARNING,
            "message": "Step skipped because it is disabled.",
            "duration_seconds": duration_seconds,
            "critical": is_critical,
        }

    try:

        write_pipeline_log(
            log_id=step_log_id,
            execution_id=JOB_RUN_ID,
            notebook_name=NOTEBOOK_NAME,
            layer_name=step_layer,
            entity_name=step_entity,
            target_table="not_applicable",
            status=status,
            message=message,
            started_at=started_at,
            finished_at=None,
            duration_seconds=None,
            records_read=None,
            records_written=None,
        )

        log_info(
            pipeline_logger=logger,
            message=f"Starting step: {step_name}",
        )

        dbutils.notebook.run(
            notebook_path,
            timeout_seconds=0,
        )

        finished_at = datetime.now()
        duration_seconds = (
            finished_at - started_at
        ).total_seconds()

        write_pipeline_log(
            log_id=str(uuid.uuid4()),
            execution_id=JOB_RUN_ID,
            notebook_name=NOTEBOOK_NAME,
            layer_name=step_layer,
            entity_name=step_entity,
            target_table="not_applicable",
            status=EXECUTION_STATUS_SUCCESS,
            message=f"Step completed successfully: {step_name}",
            started_at=started_at,
            finished_at=finished_at,
            duration_seconds=duration_seconds,
            records_read=None,
            records_written=None,
        )

        log_success(
            pipeline_logger=logger,
            message=f"Step completed: {step_name}",
        )

        return {
            "step_name": step_name,
            "status": EXECUTION_STATUS_SUCCESS,
            "message": "Step completed successfully.",
            "duration_seconds": duration_seconds,
            "critical": is_critical,
        }

    except Exception as error:

        finished_at = datetime.now()
        duration_seconds = (
            finished_at - started_at
        ).total_seconds()

        error_message = str(error)
        error_stacktrace = traceback.format_exc()

        write_pipeline_log(
            log_id=str(uuid.uuid4()),
            execution_id=JOB_RUN_ID,
            notebook_name=NOTEBOOK_NAME,
            layer_name=step_layer,
            entity_name=step_entity,
            target_table="not_applicable",
            status=EXECUTION_STATUS_FAILED,
            message=f"Step failed: {step_name} | {error_message}",
            started_at=started_at,
            finished_at=finished_at,
            duration_seconds=duration_seconds,
            records_read=None,
            records_written=None,
        )

        log_error(
            pipeline_logger=logger,
            message=f"Step failed: {step_name}",
            error=error,
        )

        return {
            "step_name": step_name,
            "status": EXECUTION_STATUS_FAILED,
            "message": error_message,
            "duration_seconds": duration_seconds,
            "critical": is_critical,
            "stacktrace": error_stacktrace,
        }

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Execute Pipeline Steps

# COMMAND ----------

pipeline_results = []

pipeline_started_at = datetime.now()

for step_config in PIPELINE_STEPS:

    step_result = run_notebook_step(
        step_config=step_config,
    )

    pipeline_results.append(step_result)

    if (
        step_result["status"] == EXECUTION_STATUS_FAILED
        and step_result["critical"]
        and FAIL_ON_STEP_ERROR
    ):

        raise Exception(
            f"Critical pipeline step failed: "
            f"{step_result['step_name']} | "
            f"{step_result['message']}"
        )

pipeline_finished_at = datetime.now()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Pipeline Summary

# COMMAND ----------

total_steps = len(pipeline_results)

success_steps = len([
    result
    for result in pipeline_results
    if result["status"] == EXECUTION_STATUS_SUCCESS
])

warning_steps = len([
    result
    for result in pipeline_results
    if result["status"] == EXECUTION_STATUS_WARNING
])

failed_steps = len([
    result
    for result in pipeline_results
    if result["status"] == EXECUTION_STATUS_FAILED
])

pipeline_duration_seconds = (
    pipeline_finished_at - pipeline_started_at
).total_seconds()

print("=" * 90)
print("FULL PIPELINE SUMMARY")
print("=" * 90)
print(f"Job Run ID: {JOB_RUN_ID}")
print(f"Total steps: {total_steps}")
print(f"Successful steps: {success_steps}")
print(f"Warning steps: {warning_steps}")
print(f"Failed steps: {failed_steps}")
print(f"Pipeline duration seconds: {pipeline_duration_seconds}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# FINAL JOB LOG
# ============================================================

final_status = (
    EXECUTION_STATUS_FAILED
    if failed_steps > 0
    else EXECUTION_STATUS_SUCCESS
)

write_pipeline_log(
    log_id=str(uuid.uuid4()),
    execution_id=JOB_RUN_ID,
    notebook_name=NOTEBOOK_NAME,
    layer_name=LAYER_NAME,
    entity_name=ENTITY_NAME,
    target_table=PIPELINE_LOG_TABLE,
    status=final_status,
    message=(
        f"Full pipeline finished. "
        f"success={success_steps}, "
        f"warning={warning_steps}, "
        f"failed={failed_steps}"
    ),
    started_at=pipeline_started_at,
    finished_at=pipeline_finished_at,
    duration_seconds=pipeline_duration_seconds,
    records_read=None,
    records_written=None,
)

# COMMAND ----------

if failed_steps > 0:

    print(
        f"WARNING: Full pipeline finished with "
        f"{failed_steps} failed step(s)."
    )

    if FAIL_ON_STEP_ERROR:
        raise Exception(
            f"Full pipeline failed with "
            f"{failed_steps} failed step(s)."
        )

print("FULL PIPELINE EXECUTION COMPLETED")

BRAZIL LEGISLATIVE ANALYTICS MEDALLION
01 - RUN FULL PIPELINE
Execution Timestamp: 2026-05-20 04:38:40.630501
Catalog: brazil_legislative_analytics
Environment: dev


2026-05-20 04:38:42 | INFO | JOBS | jobs.01_run_full_pipeline | Starting step: validate_project_setup
2026-05-20 04:39:15 | INFO | JOBS | jobs.01_run_full_pipeline | [SUCCESS] Step completed: validate_project_setup
2026-05-20 04:39:16 | INFO | JOBS | jobs.01_run_full_pipeline | Starting step: validate_api_connection
2026-05-20 04:39:39 | INFO | JOBS | jobs.01_run_full_pipeline | [SUCCESS] Step completed: validate_api_connection
2026-05-20 04:39:40 | WARNING | JOBS | jobs.01_run_full_pipeline | Step skipped: bronze_deputados
2026-05-20 04:39:42 | WARNING | JOBS | jobs.01_run_full_pipeline | Step skipped: bronze_frentes
2026-05-20 04:39:44 | WARNING | JOBS | jobs.01_run_full_pipeline | Step skipped: bronze_frentes_membros
2026-05-20 04:39:46 | WARNING | JOBS | jobs.01_run_full_pipeline | Step skipped: bronze_frentes_detalhes
2026-05-20 04:39:47 | WARNING | JOBS | jobs.01_run_full_pipeline | Step skipped: bronze_eventos
2026-05-20 04:39:49 | WARNING | JOBS | jobs.01_run_full_pipeline | St

FULL PIPELINE SUMMARY
Job Run ID: b6d8a5d3-16d2-4f5e-b413-c6ac3ff8fb8c
Total steps: 53
Successful steps: 6
Warning steps: 47
Failed steps: 0
Pipeline duration seconds: 230.696849
FULL PIPELINE EXECUTION COMPLETED


PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: df48d370-e1b5-4bef-9014-0befa8f7c733


utils_config loaded successfully.


utils_table_logger loaded successfully.
